In [5]:
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the cleaned dataset
df = pd.read_csv("../../data/processed/nvidia_articles_cleaned.csv")

# Quick check
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print()
print(df[["source", "title", "word_count", "relevance_score"]].head(5))
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
# Pick the first article
sample_text = df["full_text_clean"].iloc[0]

print("Article title:", df["title"].iloc[0])
print("Article length:", len(sample_text), "characters")
print()

# Chunk it using LangChain
sample_chunks = splitter.split_text(sample_text)

print("Number of chunks produced:", len(sample_chunks))
print()

# Inspect first two chunks
for i, chunk in enumerate(sample_chunks[:2]):
    print(f"--- Chunk {i} ---")
    print(chunk)
    print("Characters:", len(chunk))
    print()
    # This list will collect every chunk row
all_chunks = []

# Loop over every article in the dataframe
for doc_id, row in df.iterrows():
    
    text = row["full_text_clean"]
    
    # Skip if text is empty or not a string
    if not isinstance(text, str) or len(text.strip()) == 0:
        continue
    
    # Chunk this article using LangChain
    chunks = splitter.split_text(text)
    
    # For each chunk, build one row of metadata
    for chunk_num, chunk_text in enumerate(chunks):
        
        chunk_row = {
            "doc_id":           doc_id,
            "chunk_id":         f"doc_{doc_id}_chunk_{chunk_num}",
            "source":           row["source"],
            "title":            row["title"],
            "url":              row["url"],
            "published":        row["published"],
            "matched_keywords": row["matched_keywords"],
            "relevance_score":  row["relevance_score"],
            "chunk_text":       chunk_text,
            "char_count":       len(chunk_text)
        }
        
        all_chunks.append(chunk_row)

# Convert the list of rows into a DataFrame
df_chunks = pd.DataFrame(all_chunks)

print("Total chunks created:", len(df_chunks))
print("Chunks per article on average:", round(len(df_chunks) / len(df), 1))
print()
print(df_chunks.head(3))

Shape: (122, 11)
Columns: ['source', 'title', 'url', 'published', 'summary', 'collected_at', 'full_text', 'word_count', 'full_text_clean', 'matched_keywords', 'relevance_score']

            source                                              title  \
0  NVIDIA Newsroom  NVIDIA Brings Trusted, 24/7 AI Agents to Telec...   
1  NVIDIA Newsroom  NAIRR Science Program Reshapes Scientific Rese...   
2  NVIDIA Newsroom  From Materials Simulation to Experimental Astr...   
3  NVIDIA Newsroom  NVIDIA Vera CPU Opens the Way for Agentic Scie...   
4  NVIDIA Newsroom  Eco Wave Power Turns Waves Into Watts With NVI...   

   word_count  relevance_score  
0         979                8  
1         843                3  
2        1013                6  
3         504                2  
4         819                5  
Article title: NVIDIA Brings Trusted, 24/7 AI Agents to Telecom Operations
Article length: 7027 characters

Number of chunks produced: 9

--- Chunk 0 ---
Telecom operators have seen re

In [7]:
import os

output_path = "../../data/chunked/nvidia_article_chunks_langchain_claude.csv"
df_chunks.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Total rows saved:", len(df_chunks))

Saved to: ../../data/chunked/nvidia_article_chunks_langchain_claude.csv
Total rows saved: 1600
